
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
from langchain.chat_models import init_chat_model
llm= init_chat_model("ollama:llama3.2:latest")

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The release year of the movie")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")



In [5]:
model_with_structured_output=llm.with_structured_output(Movie)
response=model_with_structured_output.invoke("Tell me about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

### Message ouptut along with the Parsed Structure

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = llm.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response


{'raw': AIMessage(content='{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.5}\n\n   ', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-27T15:33:15.4653846Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3076310800, 'load_duration': 417256000, 'prompt_eval_count': 32, 'prompt_eval_duration': 263884800, 'eval_count': 32, 'eval_duration': 1568731200, 'model_name': 'llama3.2:latest'}, id='lc_run--019dcf92-b83d-7c23-a95b-db660360139c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 32, 'total_tokens': 64}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5),
 'parsing_error': None}

### Nested Structure

In [7]:
class Actor(BaseModel):
    """An actor with details."""
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role played by the actor in the movie")
class MovieDetails(BaseModel):
    """Detailed information about a movie."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    cast: list[Actor] = Field(..., description="List of actors and their roles in the movie")
    budget:float |None=Field(None,description="The budget of the movie in millions USD")



In [8]:
model_with_nested_structure = llm.with_structured_output(MovieDetails)
response = model_with_nested_structure.invoke("Provide detailed information about the movie Inception, including its cast and budget.")
response


MovieDetails(title='Inception (2010) - A Mind-Bending Sci-Fi Action Film', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Dileep Rao', role="Arthur's Wife"), Actor(name='Clive Owen', role='Mal'), Actor(name='Marion Cotillard', role='Mal')], budget=160.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [13]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=llm.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'title': 'Avengers', 'year': 2012, 'director': 'Joss Whedon', 'rating': 8.1}

In [14]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = llm.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action, Adventure, Sci-Fi'],
 'budget': 220000000}

In [15]:
model_with_structure

RunnableBinding(bound=ChatOllama(model='llama3.2:latest'), kwargs={'format': {'title': 'MovieDetails', 'description': "dict() -> new empty dictionary\ndict(mapping) -> new dictionary initialized from a mapping object's\n    (key, value) pairs\ndict(iterable) -> new dictionary initialized as if via:\n    d = {}\n    for k, v in iterable:\n        d[k] = v\ndict(**kwargs) -> new dictionary initialized with the name=value pairs\n    in the keyword argument list.  For example:  dict(one=1, two=2)", 'type': 'object', 'properties': {'title': {'type': 'string'}, 'year': {'type': 'integer'}, 'cast': {'type': 'array', 'items': {'description': "dict() -> new empty dictionary\ndict(mapping) -> new dictionary initialized from a mapping object's\n    (key, value) pairs\ndict(iterable) -> new dictionary initialized as if via:\n    d = {}\n    for k, v in iterable:\n        d[k] = v\ndict(**kwargs) -> new dictionary initialized with the name=value pairs\n    in the keyword argument list.  For example